# Day 20 – RAG Chatbot Project – Extensive Solutions

Complete implementation of a PDF chatbot with memory, source display, file upload, and deployment.

In [ ]:
# !pip install langchain chromadb openai pypdf gradio

## Exercise 1: File uploader – add PDF on the fly

We'll extend the Gradio app to accept a PDF file, add it to the vector store, and answer questions.

In [ ]:
import os
import tempfile
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.chat_models import ChatOpenAI
import gradio as gr

# Global vectorstore
embeddings = OpenAIEmbeddings()
vectorstore = None
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

def add_pdf(file):
    global vectorstore
    # Save uploaded file temporarily
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
        tmp_file.write(file.read())
        tmp_path = tmp_file.name
    # Load and split
    loader = PyPDFLoader(tmp_path)
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(docs)
    # Add to vectorstore
    if vectorstore is None:
        vectorstore = Chroma.from_documents(chunks, embeddings)
    else:
        vectorstore.add_documents(chunks)
    os.unlink(tmp_path)
    return f"Added {len(chunks)} chunks from {file.name}"

def chat(message, history):
    if vectorstore is None:
        return "Please upload a PDF first."
    # Create chain each time (or reuse)
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm,
        retriever=vectorstore.as_retriever(),
        memory=memory
    )
    result = qa_chain({"question": message})
    return result["answer"]

# Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("# PDF Chatbot")
    with gr.Row():
        file_input = gr.File(label="Upload PDF", type="binary")
        status = gr.Textbox(label="Status")
    file_input.change(add_pdf, inputs=file_input, outputs=status)
    chatbot = gr.ChatInterface(fn=chat)

# demo.launch()  # Uncomment to run

## Exercise 2: Display source documents

Modify the chain to return source documents and show them in the UI.

In [ ]:
def chat_with_sources(message, history):
    if vectorstore is None:
        return "Please upload a PDF first."
    # Custom chain that returns sources
    from langchain.chains import RetrievalQA
    qa = RetrievalQA.from_chain_type(
        llm,
        retriever=vectorstore.as_retriever(),
        return_source_documents=True
    )
    result = qa({"query": message})
    answer = result["result"]
    sources = list(set([doc.metadata.get("source", "unknown") for doc in result["source_documents"]]))
    return f"{answer}\n\n📄 Sources: {', '.join(sources)}"

# Then use this function in Gradio ChatInterface

## Exercise 3: Deploy to Hugging Face Spaces

Create a `app.py` and `requirements.txt` and push to a Space.

In [ ]:
# app.py content (to be uploaded to HF Spaces)
app_code = """
import gradio as gr
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
import os

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# ... rest of the code ...
"""
print("See HF Spaces documentation for deployment.")

## Exercise 4: Switch to local models

Replace OpenAI with Ollama and local embeddings.

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import Ollama

local_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
local_llm = Ollama(model="llama3", temperature=0)

# Then use local_llm and local_embeddings in the same chains
vectorstore_local = Chroma.from_documents(chunks, local_embeddings)
qa_local = RetrievalQA.from_chain_type(local_llm, retriever=vectorstore_local.as_retriever())
print("Local RAG ready.")

## Bonus: Add !clear command to reset memory

Implement a custom command in the chat function.

In [ ]:
def chat_with_commands(message, history):
    if message.lower() == "!clear":
        memory.clear()
        return "Memory cleared."
    # ... normal RAG ...
    return answer

print("Add command handling to the chat function.")